# 06 — Job versions: эволюция логики одного job'а

OpenLineage-Spark формирует имя job из `{app_name}.{action_or_output_table}` (точная форма зависит от Spark-операции и версии OL). Если держать стабильный `app_name` (`lineage_06_job_versions`) и писать в одну и ту же таблицу (`default.customer_summary`), Marquez увидит **один job**, но с разным логическим планом → **job versions**.

Каждая ячейка ниже — другая версия SQL для того же output'а. Между запусками jobVersion будет меняться.

Prerequisite: прогнаны `00_setup.ipynb` и `03_aggregations.ipynb` (нужна `agg_customer_stats`). **Restart Jupyter kernel** перед запуском.

In [1]:
try:
    spark.stop()
except NameError:
    pass

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName('lineage_06_job_versions')
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.appName == 'lineage_06_job_versions', \
    f'appName fallback to {spark.sparkContext.appName!r} — restart Jupyter kernel'
print('app name:', spark.sparkContext.appName)
spark.sql('DROP TABLE IF EXISTS default.customer_summary')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/27 13:11:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/27 13:11:19 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
26/05/27 13:11:29 WARN BlackbirdModule: Unable to find Java 9+ MethodHandles.privateLookupIn.  Blackbird is not performing optimally!
26/05/27 13:11:29 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.
26/05/27 13:11:29 WARN SparkApplicationNameApplicationJobNameProvider: Failed to obtain the application name. Using the default value [unknown]. Set the [spark.app.name] or [spark.openlineage.appName] property.


app name: lineage_06_job_versions


26/05/27 13:11:33 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/27 13:11:33 WARN HiveConf: HiveConf of name hive.log.dir does not exist
26/05/27 13:11:33 WARN HiveConf: HiveConf of name hive.root.logger does not exist
26/05/27 13:11:33 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

## Job version 1: минимум — id, name

In [2]:
df_v1 = spark.sql('''
    SELECT id, name
    FROM default.raw_customers
''')
df_v1.write.mode('overwrite').saveAsTable('default.customer_summary')
print('v1 rows:', spark.table('default.customer_summary').count())

26/05/27 13:11:33 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:33 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:34 WARN OutputStatisticsOutputDatasetFacetBuilder: No jobId found in context
26/05/27 13:11:34 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:34 WARN InputFieldsCollector: Could not extract dataset identifier from org.apache.spark.sql.catalyst.analysis.ResolvedIdentifier
26/05/27 13:11:38 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
[Stage 1:>                                                          (0 + 1) / 2]

v1 rows: 100


## Job version 2: + upper(country) → country_code

In [3]:
df_v2 = spark.sql('''
    SELECT
        id,
        name,
        upper(country) AS country_code
    FROM default.raw_customers
''')
df_v2.write.mode('overwrite').saveAsTable('default.customer_summary')
print('v2 rows:', spark.table('default.customer_summary').count())

v2 rows: 100


## Job version 3: + JOIN с agg_customer_stats — total_spent

In [4]:
df_v3 = spark.sql('''
    SELECT
        c.id,
        c.name,
        upper(c.country)            AS country_code,
        coalesce(s.total_spent, 0)  AS total_spent
    FROM default.raw_customers c
    LEFT JOIN default.agg_customer_stats s ON s.customer_id = c.id
''')
df_v3.write.mode('overwrite').saveAsTable('default.customer_summary')
print('v3 rows:', spark.table('default.customer_summary').count())
spark.table('default.customer_summary').show(5, truncate=False)

26/05/27 13:11:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6
26/05/27 13:11:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6
26/05/27 13:11:40 ERROR ContextFactory: Query execution is null: can't emit event for executionId 6
                                                                                

v3 rows: 100


26/05/27 13:11:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9
26/05/27 13:11:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9
26/05/27 13:11:42 ERROR ContextFactory: Query execution is null: can't emit event for executionId 9


+---+------+------------+------------------+
|id |name  |country_code|total_spent       |
+---+------+------------+------------------+
|0  |user_0|RU          |2810.926834750286 |
|1  |user_1|US          |1873.2587614025167|
|2  |user_2|DE          |2645.14552754322  |
|3  |user_3|FR          |1151.0686958279296|
|4  |user_4|RU          |2655.2608749794836|
+---+------+------------+------------------+
only showing top 5 rows



In [6]:
spark.stop()

## Что смотреть в Marquez

1. UI → jobs (namespace `hadoop-cluster`) → найти job с именем, содержащим `customer_summary` (полное имя зависит от Spark action; OL-Spark обычно использует `execute_create_data_source_table_as_select_command` или подобное в качестве суффикса). Способ найти через API:
   ```bash
   curl -s 'http://localhost:5000/api/v1/namespaces/hadoop-cluster/jobs' \
     | jq '.jobs[] | select(.name | contains("customer_summary")) | .name'
   ```
2. В detail job'а — tab **Versions**: должно быть 3 версии, у каждой свой план и свой column lineage facet.
3. Версия 3 будет иметь 2 input dataset'а (`raw_customers` + `agg_customer_stats`), версии 1-2 — только `raw_customers`.